In [1]:
# GOAL : Clinical Trial Matching with LLM on backend for querying
# difficulty: ensuring client aiming to search for clinical trials are eligible / trial is fitting and feasible 

In [2]:
import requests

url = "https://clinicaltrials.gov/api/v2/studies"
params = {
    "query.cond": "type 2 diabetes",
    "filter.overallStatus": "RECRUITING",
    "pageSize": 1,
    "format": "json",
}

In [3]:
response = requests.get(url, params=params)
print(response.status_code) 
data = response.json()

200


In [4]:
import json, os

SNAPSHOT = "trials_snapshot.json"

if os.path.exists(SNAPSHOT):            # ifff snapshot already saved → load it
    with open(SNAPSHOT) as f:
        data = json.load(f)
else:                                   # first time only → fetch 100 trials + save them
    params["pageSize"] = 100
    data = requests.get(url, params=params).json()
    with open(SNAPSHOT, "w") as f:
        json.dump(data, f)

print(len(data["studies"]), "trials loaded")

100 trials loaded


In [5]:
# let's explore the actual clinical trial data that is publically available here
data.keys()

dict_keys(['studies', 'nextPageToken'])

In [6]:
data["studies"][0].keys()

dict_keys(['protocolSection', 'derivedSection', 'hasResults'])

note to self: protocolSection will have the info ab the trial [by whoever registered] - so info on title, eligibility, locations, etc. 
derivedSection: extra info specifically ClinicalTrials.gov adds like condition categories, etc.
hasResults: True/False - question: does trial have posted results? core! for checking if a finished study has posted its final results 

In [7]:
data["studies"][0]["protocolSection"].keys()

dict_keys(['identificationModule', 'statusModule', 'sponsorCollaboratorsModule', 'oversightModule', 'descriptionModule', 'conditionsModule', 'designModule', 'armsInterventionsModule', 'outcomesModule', 'eligibilityModule', 'contactsLocationsModule', 'ipdSharingStatementModule'])

In [8]:
#need to search for main criteria for eligibility and feasibility of trial
#here: NCT ID, title, eligibility criteria , minimum age, etc. 
# let's look into eligibilityModule, identificationModule, & conditionsModule for now
print("Eligibility Module has", data["studies"][0]["protocolSection"]["eligibilityModule"].keys())
print("Identification Module has", data["studies"][0]["protocolSection"]["identificationModule"].keys())
print("Conditions Module has", data["studies"][0]["protocolSection"]["conditionsModule"].keys())

Eligibility Module has dict_keys(['eligibilityCriteria', 'healthyVolunteers', 'sex', 'minimumAge', 'maximumAge', 'stdAges'])
Identification Module has dict_keys(['nctId', 'orgStudyIdInfo', 'organization', 'briefTitle', 'officialTitle'])
Conditions Module has dict_keys(['conditions', 'keywords'])


In [9]:
print("Alright, NCT ID is", data["studies"][0]["protocolSection"]["identificationModule"]["nctId"], "and the title is ", data["studies"][0]["protocolSection"]["identificationModule"]["briefTitle"], \
      "or more formally", data["studies"][0]["protocolSection"]["identificationModule"]["officialTitle"])

Alright, NCT ID is NCT07502508 and the title is  Phase 2 Trial of Icovamenib in Participants With Type 2 Diabetes Mellitus Who Are Not Achieving Glycemic Targets While Using GLP-1-Based Therapy or more formally Phase 2 Randomized, Double-blind, Placebo-Controlled Trial Assessing the Efficacy and Safety of Icovamenib in Participants With Type 2 Diabetes Not Achieving Glycemic Targets Despite GLP-1-Based Therapy


In [10]:
print(data["studies"][0]["protocolSection"]["eligibilityModule"]["eligibilityCriteria"])
print("Allowed Sex:", data["studies"][0]["protocolSection"]["eligibilityModule"]["sex"])
print('Minimum Age for Study:', data["studies"][0]["protocolSection"]["eligibilityModule"]["minimumAge"])

Key Inclusion Criteria:

1. Males or females, age ≥18 years and ≤75 years
2. Have been diagnosed with T2D
3. Taking FDA-approved GLP-1 based therapy (administered as subcutaneous injection) and have been treated with lifestyle management and 0 to 2 additional antihyperglycemic medications (metformin and/or SGLT2 inhibitor) with a stable dose of all medications for at least 3 months prior to screening
4. Have HbA1c ≥7.0 and ≤10.5%
5. Have a BMI 25 to 45 kg/m2
6. Female participants of childbearing potential must have a negative pregnancy test, must be non-lactating, must be willing to have additional pregnancy tests during the study, must agree to the sex and contraception requirements.
7. Willing and able to provide written, signed informed consent and be willing and able to comply with all study procedures and tests.

Key Exclusion Criteria:

1. Have type 1 diabetes mellitus or a secondary form of diabetes
2. Have a history of diabetic ketoacidosis or hyperosmolar coma in the 6 months

In [11]:
def extract_trial(study):
    p = study["protocolSection"]
    return {
        "nct_id": p["identificationModule"]["nctId"],
        "title":p["identificationModule"]["briefTitle"],
        "conditions": p.get("conditionsModule",{}).get("conditions", ["No Conditions Listed"]),
        "eligibility": p.get("eligibilityModule", {}).get("eligibilityCriteria", "No Eligibility Criteria Listed"),
        "min_age": p.get("eligibilityModule", {}).get("minimumAge", "No Minimum Age Listed"), # came back to add fallbacks for many conditions that may not be in every study
        "max_age": p.get("eligibilityModule",{}).get("maximumAge", "No Age Limit"),  # alright so many of the studies have no max age cap
        "sex": p.get("eligibilityModule",{}).get("sex", "Sex Not Specified"),
    }

In [12]:
extract_trial(data["studies"][0])

{'nct_id': 'NCT07502508',
 'title': 'Phase 2 Trial of Icovamenib in Participants With Type 2 Diabetes Mellitus Who Are Not Achieving Glycemic Targets While Using GLP-1-Based Therapy',
 'conditions': ['Type 2 Diabetes'],
 'eligibility': 'Key Inclusion Criteria:\n\n1. Males or females, age ≥18 years and ≤75 years\n2. Have been diagnosed with T2D\n3. Taking FDA-approved GLP-1 based therapy (administered as subcutaneous injection) and have been treated with lifestyle management and 0 to 2 additional antihyperglycemic medications (metformin and/or SGLT2 inhibitor) with a stable dose of all medications for at least 3 months prior to screening\n4. Have HbA1c ≥7.0 and ≤10.5%\n5. Have a BMI 25 to 45 kg/m2\n6. Female participants of childbearing potential must have a negative pregnancy test, must be non-lactating, must be willing to have additional pregnancy tests during the study, must agree to the sex and contraception requirements.\n7. Willing and able to provide written, signed informed cons

In [13]:
# alright cool we have a lot of the data we want to extract from each trial, now let us have a df with many trials of interest
# will narrow down to a few conditions of interest for now
import pandas as pd

all_trials = [ ]

for study in data["studies"]:
    all_trials.append(extract_trial(study))      ##function from b4

df = pd.DataFrame(all_trials)

In [14]:
# cool, let us convert this to text now for model to read
def trial_to_text(row):
    text = (f"Trial {row['nct_id']}: {row['title']}\n"
            f"Conditions: {','.join(row['conditions'])}\n"
            f"Ages: {row['min_age']} to {row['max_age']}\n"
            f"Sex: {row['sex']}\n"
            f"Eligibility Conditions: {row['eligibility']}\n"
    )
    return text


In [15]:
print(trial_to_text(df.iloc[0]))

Trial NCT07502508: Phase 2 Trial of Icovamenib in Participants With Type 2 Diabetes Mellitus Who Are Not Achieving Glycemic Targets While Using GLP-1-Based Therapy
Conditions: Type 2 Diabetes
Ages: 18 Years to 75 Years
Sex: ALL
Eligibility Conditions: Key Inclusion Criteria:

1. Males or females, age ≥18 years and ≤75 years
2. Have been diagnosed with T2D
3. Taking FDA-approved GLP-1 based therapy (administered as subcutaneous injection) and have been treated with lifestyle management and 0 to 2 additional antihyperglycemic medications (metformin and/or SGLT2 inhibitor) with a stable dose of all medications for at least 3 months prior to screening
4. Have HbA1c ≥7.0 and ≤10.5%
5. Have a BMI 25 to 45 kg/m2
6. Female participants of childbearing potential must have a negative pregnancy test, must be non-lactating, must be willing to have additional pregnancy tests during the study, must agree to the sex and contraception requirements.
7. Willing and able to provide written, signed inform

In [16]:
#apply function to entire df
df["text"] = df.apply(trial_to_text, axis = 1)
#test
print(df["text"].iloc[5])

Trial NCT06279637: Joint Patient and Caregiver Intervention for Older African Americans With Type 2 Diabetes
Conditions: Type 2 Diabetes
Ages: 50 Years to 120 Years
Sex: ALL
Eligibility Conditions: Patient Inclusion Criteria:

1. Age ≥50 years of age;
2. Self-identified as Black/African American;
3. Clinical diagnosis of T2DM and poorly controlled, defined as HbA1c ≥8% at the screening visit;
4. Able to communicate in English; and
5. Has an informal caregiver (family member or close friend willing to participate in the study for 6 months.

Caregiver Inclusion Criteria:

1. Willing to attend 4, one-hour sessions with the study participant;
2. Willing to support the study participant for study duration (6 months); and
3. Willing to complete brief baseline, 3- and 6-month assessments.

Patient/Caregiver Exclusion Criteria:

1. Mental confusion at screening assessment suggesting significant dementia;
2. Alcohol or drug abuse/dependency at screening assessment;
3. Active psychosis or acute 

let's import miniLM so we care able to query the data with similar words

In [17]:
#pip install sentence_transformers

In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1117 18:14:43.656000 7977 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10247.72it/s]


In [19]:
list_of_trials = df["text"].tolist()
embeddings = model.encode(list_of_trials)
print(embeddings.shape)

(100, 384)


In [20]:
df["text"].str.len().describe()

count      100.000000
mean      2374.690000
std       2219.893103
min        261.000000
25%       1071.750000
50%       1693.000000
75%       2379.500000
max      14148.000000
Name: text, dtype: float64

okay so the model I imported reads roughly 256 tokens:

typically 1 token ≈ 4 characters, so about... 1,000 characters

distribution of our df definitely surpasses this, so let us split each trial into chunks and embed upon those chunks separately, 
or we would face an issue where everything past 256 tokens would get silently ignored

In [21]:
print(df["eligibility"].str.contains(r"\* ").mean())
print(df["eligibility"].str.contains("Exclusion Criteria").mean())

0.73
0.91


typically, we would split by # of characters w/ added overlap incase something is cut-off, then assign IDs to the chunks so we know where everything came from

but here, most (94% from above) are separated into * lists 

so we default to chunking by separating into bullets, then fall-back on splitting every 1000 characters if there are no *'s or if the bullets >1000 characters

In [22]:
crits = df["eligibility"].str.split(r"\* ").explode()
crits.str.len().describe()
(crits.str.len() > 800).sum()

np.int64(46)

In [23]:
def split_stars(text):
    pieces = text.split("* ")            # cut the text wherever "* " appears → list of strings
    pieces = [p.strip() for p in pieces] # remove extra spaces/newlines from each piece
    pieces = [p for p in pieces if p]    # drop empty strings ("" counts as False)
    return pieces

In [24]:
def split_paragraphs(text):
    pieces = text.split("\n\n") #splitting at blank line
    pieces = [p.strip() for p in pieces] 
    pieces = [p for p in pieces if p]  
    return pieces

In [25]:
import re
def split_numbers(text):
    pieces = re.split(r"\n\s*\d+\.\s+", text)
    pieces = [p.strip() for p in pieces] 
    pieces = [p for p in pieces if p] 
    return pieces

In [26]:
def fixed_size_split(text, size=800, overlap=100):
    pieces = []
    start = 0
    while start < len(text):
        end = start + size
        pieces.append(text[start:end])
        start = end - overlap
    return pieces

In [27]:
#ok let's actually apply this to split_text def
# we made the rules but lets make the def for actually using this for our text!
LIMIT = 1100   #256 tokens × ~4.6 chars ≈ 1,170
#minus a safety margin; header + label + piece must fit

def split_text(text, methods, budget):
    #1 its already short enough is case one, don't need to chunk
    if len(text) <= budget:
        return [text]

    #2 no methods left (remember this keeps running)
    # so no methods left? use fixed-size cutter
    if len(methods) == 0:
        return fixed_size_split(text, size = budget)

    #3 try the first method in the list!
    pieces = methods[0](text)

    #4 if nothing split --> next method on same text fr
    if len(pieces) == 1:
        return split_text(text, methods[1:], budget)

    #5 or... check each piece + split further if needed
    results = []
    for piece in pieces:
        results.extend(split_text(piece, methods[1:], budget))
    return results

methods = [split_stars, split_numbers, split_paragraphs]

In [28]:
# well let us test on some pieces...
pieces = split_text(df["eligibility"].iloc[0], methods, 800)
for p in pieces[:5]:
    print(repr(p))
    print("---")

'Key Inclusion Criteria:'
---
'Males or females, age ≥18 years and ≤75 years'
---
'Have been diagnosed with T2D'
---
'Taking FDA-approved GLP-1 based therapy (administered as subcutaneous injection) and have been treated with lifestyle management and 0 to 2 additional antihyperglycemic medications (metformin and/or SGLT2 inhibitor) with a stable dose of all medications for at least 3 months prior to screening'
---
'Have HbA1c ≥7.0 and ≤10.5%'
---


ok so now it is splitting and the slpits make sense but...
each pieced does not actually say whether it is inclusion or exclusion criteria so ...
need that to be added in

In [29]:
def split_sections(eligibility):
    # splits only once when finds exclusion criteria (case-insensitive)
    parts = re.split(r"exclusion criteria:?", eligibility, maxsplit=1, flags=re.IGNORECASE)

    if len(parts) == 2:
        inclusion, exclusion = parts
        # removes inclusion criteria header 
        inclusion = re.sub(r"inclusion criteria:?", "", inclusion, flags=re.IGNORECASE)
        return [("Inclusion", inclusion.strip()), ("Exclusion", exclusion.strip())]

    # no exclusion header found
    return [("Eligibility", eligibility.strip())]

In [30]:
print(split_sections(df["eligibility"].iloc[0]))   # normal trial
print(split_sections("INCLUSION CRITERIA:\n* Age 25+\nEXCLUSION CRITERIA:\n* Type 1"))   # all caps
print(split_sections("Adults with T2D who are not pregnant."))   # no headers

[('Inclusion', 'Key \n\n1. Males or females, age ≥18 years and ≤75 years\n2. Have been diagnosed with T2D\n3. Taking FDA-approved GLP-1 based therapy (administered as subcutaneous injection) and have been treated with lifestyle management and 0 to 2 additional antihyperglycemic medications (metformin and/or SGLT2 inhibitor) with a stable dose of all medications for at least 3 months prior to screening\n4. Have HbA1c ≥7.0 and ≤10.5%\n5. Have a BMI 25 to 45 kg/m2\n6. Female participants of childbearing potential must have a negative pregnancy test, must be non-lactating, must be willing to have additional pregnancy tests during the study, must agree to the sex and contraception requirements.\n7. Willing and able to provide written, signed informed consent and be willing and able to comply with all study procedures and tests.\n\nKey'), ('Exclusion', '1. Have type 1 diabetes mellitus or a secondary form of diabetes\n2. Have a history of diabetic ketoacidosis or hyperosmolar coma in the 6 m

yay it works!

In [31]:
#ok but how much variation in length of titles is there
#because that effects what our real budget is
#let's define the entire actual header part that isnt the resulting text
def make_header(row):
    return (f"Trial {row['nct_id']}: {row['title']}\n"
            f"Conditions: {', '.join(row['conditions'])}\n"
            f"Ages: {row['min_age']} to {row['max_age']}\n"
            f"Sex: {row['sex']}\n")

In [32]:
def chunk_trial(row):
    header = make_header(row)
    budget = LIMIT - len(header) - 25 
    
    chunks = []
    for label, section_text in split_sections(row["eligibility"]):
        for piece in split_text(section_text, methods, budget):
            chunks.append({
                "nct_id": row["nct_id"],
                "section": label,
                "text": header + f"{label} criterion: " + piece,
            })
    return chunks

In [33]:
test_chunks = chunk_trial(df.iloc[0])
print(len(test_chunks))
print(test_chunks[0]["text"])
print(test_chunks[-1]["text"])

2
Trial NCT07502508: Phase 2 Trial of Icovamenib in Participants With Type 2 Diabetes Mellitus Who Are Not Achieving Glycemic Targets While Using GLP-1-Based Therapy
Conditions: Type 2 Diabetes
Ages: 18 Years to 75 Years
Sex: ALL
Inclusion criterion: Key 

1. Males or females, age ≥18 years and ≤75 years
2. Have been diagnosed with T2D
3. Taking FDA-approved GLP-1 based therapy (administered as subcutaneous injection) and have been treated with lifestyle management and 0 to 2 additional antihyperglycemic medications (metformin and/or SGLT2 inhibitor) with a stable dose of all medications for at least 3 months prior to screening
4. Have HbA1c ≥7.0 and ≤10.5%
5. Have a BMI 25 to 45 kg/m2
6. Female participants of childbearing potential must have a negative pregnancy test, must be non-lactating, must be willing to have additional pregnancy tests during the study, must agree to the sex and contraception requirements.
7. Willing and able to provide written, signed informed consent and be wi

In [34]:
header_lengths = df.apply(make_header, axis=1).str.len()
print(header_lengths.describe())

count    100.000000
mean     224.340000
std       69.005814
min      117.000000
25%      178.750000
50%      211.500000
75%      249.250000
max      471.000000
dtype: float64


alright... so the max of 559 is an issue because if we have 1170 - 702 -25 = 440 approx so 

the chunks for the trial would be way smaller than needed

In [35]:
# but let us actually check the worst-case meaning the trial with the longest header
worst = df.loc[df.apply(make_header, axis=1).str.len().idxmax()]
chunks = chunk_trial(worst)
print(worst["nct_id"], len(chunks), max(len(c["text"]) for c in chunks))

NCT05610865 2 1023


ok nice so it made the cut of being <= 1,100 so the header, label , and criteria together still fit in miniLM window

In [36]:
# with that info let us now chunk all 100 trials that we are looking at 
all_chunks = []
for _, row in df.iterrows():
    all_chunks.extend(chunk_trial(row))

chunks_df = pd.DataFrame(all_chunks)
print(len(chunks_df))                                    # total chunks
print(chunks_df["text"].str.len().max())                 # longest chunk 
print(chunks_df.groupby("nct_id").size().describe())     # chunks per trial

1194
1096
count    100.000000
mean      11.940000
std       13.519772
min        2.000000
25%        2.000000
50%        3.000000
75%       18.000000
max       53.000000
dtype: float64


hmm
ok so this is super interesting....
half of our data has 2-3 chunks... so the data of the trial was short enough for most of the data to be maintained and not be oversplit but then in the 75% + , we have a huge amount of chunks, so some trials were so long they are being split so much, even by single criteria

2 issues w this (that i notice now):

1. retrieval wise a trial w/ 53 chunks has way more chances to match a search than a trial with 2 chunks.. like imagine doing a google search and the first 30 or so searches are all from the same source [ so we should group results by trial in the retrieval step]

2. also in terms of matching score like to what a patient needs, the longer ones have more chance of scoring highly becaause there are 53 chances of them with many key words that match [ same fix]

In [37]:
# embed chunks
chunk_embeddings = model.encode(
    chunks_df["text"].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
)
print(chunk_embeddings.shape)

Batches: 100%|██████████| 38/38 [00:04<00:00,  8.68it/s]

(1194, 384)


ok let us make the search functionality now (yay)

we already turned each chunk into an embedding so text with similar menaing get similar numbers 

we score each chunk so essentially cosine similarity will tell us how similar vectors pointing in same direction are

then we rank the scores  and we also want one result per trial again like we talked ab b4 so there is the regrouping at the end 



In [38]:
# search function
def search(question, k=5):
    q = model.encode([question], normalize_embeddings=True)    # question → 1 vector
    scores = (chunk_embeddings @ q.T).flatten()                # similarity to every chunk

    results = chunks_df.copy()
    results["score"] = scores
    results = results.sort_values("score", ascending=False)   # best chunks first
    results = results.drop_duplicates("nct_id")                # each trial's best chunk only
    return results.head(k)

In [39]:
# let's try it
# pseudo patient interest question  ab if patient A
#has type 2 diabetes and takes insulin, what trials are recommended
res = search("I have type 2 diabetes and I take insulin")
for _, r in res.iterrows():
    print(r["nct_id"], round(r["score"], 3), r["section"])
    print(r["text"][-200:])
    print("---")

NCT05766735 0.584 Inclusion
hroughout the study
5. Diabetes management visit with medical provider within 12 months of screening date
6. If on weight altering medications (e.g., GLP-1, GIP), on a stable dose for about four weeks
---
NCT07659574 0.573 Exclusion
c. (except for cumulative use \<7 days with the last dose administered more than 7 half-lives prior to the first day of screening); 3) Use of weight-loss medications (including but not limited to orli
---
NCT07690163 0.563 Inclusion
onists, DPP-4 inhibitors, or Sulfonylureas) as monotherapy or combination therapy.
* Ability to provide written informed consent and willingness to adhere to the study protocol and follow-up schedule.
---
NCT06959797 0.549 Exclusion
loop for Type 2 Diabetes
Conditions: Type 2 Diabetes
Ages: 18 Years to No Age Limit
Sex: ALL
Exclusion criterion: Severe insulin resistance defined by a daily insulin requirement of more than 1.5 U/kg
---
NCT04501406 0.548 Inclusion
except if glycemic control deteriorates 

so.
a lot to discuss here, idk if we want to move on here or add some more complexity at this initial step or do a mix of both

the 2 of 5 top trials matched based on their exclusion not inclusion criteria so if we are trying to find the best match for our patient, this wouldn't be helpful....

BUT it isn't that simple because our simulated patient takes insulin but we do not know how much so the exclusion criteria listed of "Severe insulin resistance defined by a daily insulin requirement of more than 1.5 U/kg" might actually NOT apply, so this is a 'need more information' condition